# SMITH-Agent evaluation and probe feasibility

This notebook covers two auditable SMITH-Agent stages: multi-reference panel evaluation on a locked spatial test set and integrated probe-feasibility filtering. It recomputes group statistics and feasibility pass rates from pinned tables.

[Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/agent_section/05_SMITH_Agent_Evaluation_source.ipynb)

## Setup

Run this notebook from a cloned SMITH repository with `pip install -e '.[notebooks]'`. All inputs are checksum-validated before analysis.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repository(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside a SMITH repository checkout.")


ROOT = find_repository(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

from smith.reproducibility import check_case, load_cases, run_case

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
print(f"Repository: {ROOT}")


In [ ]:
CASE_ID = "05_agent"
case = load_cases()[CASE_ID]
status = check_case(case)
if status["inputs"]:
    display(pd.DataFrame(status["inputs"])[["path", "exists", "sha256_ok"]])
else:
    print("This tutorial creates its deterministic input during execution.")
assert status["ready"], "The pinned tutorial inputs are missing or have changed."

output_dir = ROOT / "outputs" / "notebooks" / CASE_ID
result = run_case(case, output_dir)
print(f"Summary written to: {result['summary_json']}")
result


## Analysis

In [ ]:
accuracy = pd.DataFrame(result["multi_reference_accuracy"])
pass_rates = pd.Series(result["feasibility_pass_rates"], name="pass_rate")
display(accuracy)
display(pass_rates.to_frame())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for panel, group in accuracy.groupby("panel"):
    axes[0].errorbar(group["panel_size"], group["mean"], yerr=group["std"],
                     marker="o", capsize=3, label=panel)
axes[0].set(xlabel="Panel size", ylabel="Cell-type accuracy", title="Multi-reference evaluation")
axes[0].legend(frameon=False, fontsize=8)
axes[1].bar(pass_rates.index, pass_rates.values, color="#b33939")
axes[1].set(ylim=(0, 1), ylabel="Fraction passing", title="Probe-feasibility gates")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()
plt.show()


## What this reproduces

The notebook recomputes multi-reference accuracy and feasibility summaries. Full Figure 6 also requires public-reference retrieval, repeated SMITH runs, a locked MERFISH test, ODT, BLAST indexes, ProbeDealer resources, and Pareto-guided hyperparameter search.